# 🚀 AI Job Search & Auto-Apply — Complete Pipeline Testing & Walkthrough
This notebook provides an in-depth, chunk-by-chunk understanding of the entire **AI Job Search & Auto-Apply** architecture.
Each chunk demonstrates and tests a standalone layer in the pipeline, from raw configuration to AI personalization, resume parsing, anti-spam validation, and headless UI verification.

---

### 📋 Complete Pipeline Chunks:
- **Chunk 1**: Environment & Module Path Resolution
- **Chunk 2**: Configuration Layer (`candidate_profile`, `email_config`, `model_config`, `smtp_config`)
- **Chunk 3**: Resume Parsing & Profile Extraction Pipeline
- **Chunk 4**: Multi-Format Recipient Ingestion, Strict Validation & Timestamped Backups
- **Chunk 5**: Recruiter Humanization & Name Inference (`Hi [Name],`)
- **Chunk 6**: Dynamic System Prompt Builder & Tone Engineering
- **Chunk 7**: AI Email Writer, Quality Validation (`EmailValidator`) & Offline Fallback Engine
- **Chunk 8**: Anti-Spam Lookback Window & Calendar Deduplication
- **Chunk 9**: RFC MIME Multipart Email Assembly & PDF Resume Attachment
- **Chunk 10**: Multi-Format Export (.csv, .xlsx, .txt, .json) & Domain-Wise Analytics
- **Chunk 11**: End-to-End Live Streaming Batch Outreach Simulation
- **Chunk 12**: Streamlit Web Application Headless Rendering Test (AppTest)


## 🧪 Chunk 1: Environment & Module Path Resolution
Ensures that the `src/` directory is present on `sys.path` so all sub-packages (`ai_email_writer`, `email_sender`, `config_loader`, `email_db_helper`) can be imported cleanly.


In [ ]:
import sys
from pathlib import Path

# Resolve project root from notebook directory
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name in ('nb', 'docs', 'doc') else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"[Chunk 1] Project Root: {PROJECT_ROOT}")
print(f"[Chunk 1] Source Directory: {SRC_DIR}")
assert SRC_DIR.is_dir(), f"Source directory {SRC_DIR} not found!"


## 🧪 Chunk 2: Configuration Layer
Reads the candidate profile, email settings, model settings, and SMTP configuration from `data/config/`.


In [ ]:
from config_loader import (
    get_candidate_profile,
    get_email_config,
    get_model_config,
    get_smtp_config,
)

profile = get_candidate_profile()
email_cfg = get_email_config()
model_cfg = get_model_config()
smtp_cfg = get_smtp_config()

print("--- Candidate Profile Summary ---")
print(f"Candidate: {profile.get('name', 'N/A')}")
print(f"Role: {profile.get('current_role', 'N/A')}")
print(f"Experience: {profile.get('experience_summary', 'N/A')}")
skills = profile.get('technical_skills', [])
print(f"Identified Skills Count: {len(skills)}")

print("\n--- Outreach Settings ---")
print(f"Subject: {email_cfg.get('subject')}")
print(f"Lookback Window: {email_cfg.get('look_back')} {email_cfg.get('look_back_format')}")
print(f"Model: {model_cfg.get('model_name')}")
print(f"SMTP Server: {smtp_cfg.get('host')}:{smtp_cfg.get('port')} (TLS={smtp_cfg.get('use_tls')})")


## 🧪 Chunk 3: Resume Parsing & Profile Extraction Pipeline
Tests PDF resume parsing and candidate profile structuring.


In [ ]:
from ai_email_writer.profile import build_candidate_profile_text

profile_text = build_candidate_profile_text()
print("[Chunk 3] Dynamic Candidate Profile Context:")
print(profile_text[:350] + "\n... [truncated for display]")
assert len(profile_text) > 50, "Candidate profile text should not be empty!"


## 🧪 Chunk 4: Multi-Format Recipient Ingestion, Strict Validation & Timestamped Backups
Extracts email addresses from arbitrary text/job descriptions, applies strict RFC regex, and tests database persistence.


In [ ]:
from email_db_helper import (
    is_valid_email,
    extract_emails_from_text,
    load_database_from_location,
    ingest_and_save_emails,
)

# 1. Validation test
assert is_valid_email("recruiter.lead@google.com") is True
assert is_valid_email("invalid@.com") is False
assert is_valid_email("fake@domain..org") is False

# 2. Text scraping test
sample_job_post = """
Hiring Senior Data Scientist!
Please send resumes to talent.acquisition@amazon.com or careers@startup.io.
Do not send to spam@internal. or fake_email.
"""
extracted = extract_emails_from_text(sample_job_post)
print(f"[Chunk 4] Extracted Valid Emails: {extracted}")
assert "talent.acquisition@amazon.com" in extracted
assert "careers@startup.io" in extracted
assert len(extracted) == 2


## 🧪 Chunk 5: Recruiter Humanization & Name Inference
Tests extracting recruiter names from email patterns (e.g. `john.doe@company.com` -> `John Doe`) while skipping generic company mailboxes.


In [ ]:
from ai_email_writer.recipient import infer_recipient_name

test_emails = [
    ("rahul.singh@capgemini.com", "Rahul Singh"),
    ("anjul.sharma01@ringg.ai", "Anjul Sharma"),
    ("hr@startup.io", None),
    ("careers@amazon.com", None),
]

print("--- Recipient Name Inference Results ---")
for email, expected in test_emails:
    inferred = infer_recipient_name(email)
    print(f"Email: {email:<35} -> Inferred: {inferred}")
    if expected is not None:
        assert inferred == expected


## 🧪 Chunk 6: Dynamic System Prompt Builder & Tone Engineering
Builds the dynamic system prompt that enforces polite tone, bulleted skill formatting (`•`), and double newline paragraph separation.


In [ ]:
from ai_email_writer.prompts import build_generic_system_prompt

sys_prompt = build_generic_system_prompt()
print("[Chunk 6] Dynamic System Prompt (First 400 chars):")
print(sys_prompt[:400] + "\n...")
assert "Polite" in sys_prompt or "Respectful" in sys_prompt or "Tone" in sys_prompt


## 🧪 Chunk 7: AI Email Writer, Quality Validation & Offline Fallback Engine
Tests `EmailValidator` for defects (placeholders, missing signatures, word count) and tests offline fallback formatting.


In [ ]:
from ai_email_writer.validator import EmailValidator
from ai_email_writer.formatter import format_email_body

validator = EmailValidator()

# Test 1: Complete and valid email
good_email = """Hi Sarah,

I am writing to express my interest in the AI Engineer role at your firm.
With 3 years of experience developing machine learning and generative AI architectures, I am eager to contribute.

Best regards,
Rahul Singh"""

val_good = validator.validate(good_email)
print(f"Valid Email Check: Passed={val_good.is_valid}")
assert val_good.is_valid is True

# Test 2: Incomplete email with placeholder leaks
bad_email = "Hey check this out. [Insert Company Name] is cool. TODO: add skills."
val_bad = validator.validate(bad_email)
print(f"Defective Email Check: Passed={val_bad.is_valid}, Issues Identified={len(val_bad.issues)}")
assert val_bad.is_valid is False

# Test 3: Fallback formatting engine
fallback_formatted = format_email_body(
    body_text="I have attached my resume for your consideration.",
    recipient_name="Sarah Connor",
    candidate_name="Rahul Singh",
)
print("\n--- Formatted Fallback Email ---")
print(fallback_formatted)
assert fallback_formatted.startswith("Hi Sarah Connor,")


## 🧪 Chunk 8: Anti-Spam Lookback Window & Calendar Deduplication
Calculates cutoff date to ensure contacts emailed within the lookback window are never spammed twice.


In [ ]:
from datetime import datetime, timedelta
from email_sender.date_utils import get_lookback_date

today = datetime.now().date()
one_month_ago = get_lookback_date(look_back=1, look_back_format="months")
two_weeks_ago = today - timedelta(days=14)
two_months_ago = today - timedelta(days=60)

print(f"Today: {today}")
print(f"Lookback Cutoff (1 month): {one_month_ago}")

# Contacted 2 weeks ago is inside lookback window -> MUST SKIP
should_skip = two_weeks_ago >= one_month_ago
print(f"Email sent 2 weeks ago ({two_weeks_ago}) skipped: {should_skip}")
assert should_skip is True

# Contacted 2 months ago is outside lookback window -> OK TO SEND
should_not_skip = two_months_ago >= one_month_ago
print(f"Email sent 2 months ago ({two_months_ago}) skipped: {should_not_skip}")
assert should_not_skip is False


## 🧪 Chunk 9: RFC MIME Multipart Email Assembly & PDF Resume Attachment
Tests building standard MIME multipart messages with headers, UTF-8 text, and attachments.


In [ ]:
from email_sender.builder import build_email_message

message = build_email_message(
    sender_email="candidate@gmail.com",
    recipient_email="recruiter@company.com",
    subject="Application: AI Engineer - Rahul Singh",
    body="Hi Recruiter,\n\nPlease review my attached resume.\n\nBest regards,\nRahul Singh",
    attachments=None,
)

print(f"[Chunk 9] MIME Message Assembled:")
print(f"   From: {message['From']}")
print(f"   To:   {message['To']}")
print(f"   Subj: {message['Subject']}")
assert message['To'] == "recruiter@company.com"


## 🧪 Chunk 10: Multi-Format Export (.csv, .xlsx, .txt, .json) & Domain-Wise Analytics
Tests exporting database to all 4 formats and grouping records by company domain.


In [ ]:
from email_db_helper import (
    export_email_database,
    group_emails_by_domain,
    get_domain_statistics,
    delete_domain_from_database,
)

mock_db = {
    "alex@google.com": "2026-03-01",
    "brian@google.com": "Never",
    "clara@microsoft.com": "2026-02-15",
    "david@startup.io": "Never",
}

# 1. Multi-format exports
csv_bytes = export_email_database(mock_db, "csv")
xlsx_bytes = export_email_database(mock_db, "xlsx")
txt_bytes = export_email_database(mock_db, "txt")
json_bytes = export_email_database(mock_db, "json")

print(f"Export Sizes: CSV={len(csv_bytes)}B, XLSX={len(xlsx_bytes)}B, TXT={len(txt_bytes)}B, JSON={len(json_bytes)}B")

# 2. Domain grouping and analytics
stats = get_domain_statistics(mock_db)
print(f"Domain Analytics: Total={stats['total_emails']}, Domains={stats['total_domains']}, Contacted={stats['contacted_count']}")
assert stats['total_domains'] == 3

# 3. Domain deletion
updated_db, deleted_count = delete_domain_from_database(mock_db, "google.com")
print(f"Domain Deletion: Removed {deleted_count} contacts from @google.com. Remaining: {len(updated_db)}")
assert deleted_count == 2
assert "alex@google.com" not in updated_db


## 🧪 Chunk 11: End-to-End Live Streaming Batch Outreach Simulation
Tests the streaming batch generator that feeds real-time updates to the Streamlit UI.


In [ ]:
from email_sender.batch import run_email_batch_stream

test_batch = ["test.lead1@domain.com", "test.lead2@domain.com"]

stream = run_email_batch_stream(
    list_of_emails=test_batch,
    subject="Senior Data Scientist Outreach",
    body="Hi,\n\nPlease review my application for the role.\n\nBest regards,\nRahul Singh",
    sender_email="candidate@gmail.com",
    sender_password="mock_test_password",
    attachments=None,
    look_back=1,
    look_back_format="months",
)

# Grab the first initialization event
init_event = next(stream)
print(f"[Chunk 11] Stream Initialized Successfully: {init_event}")
assert init_event["event"] == "init"
assert init_event["total_emails_in_db"] == 2


## 🧪 Chunk 12: Streamlit Web Application Headless Rendering Test (AppTest)
Verifies that the main Streamlit application (`bin/app.py`) initializes, renders all tabs, and executes with zero exceptions.


In [ ]:
from streamlit.testing.v1 import AppTest

app_entry = PROJECT_ROOT / "bin" / "app.py"
print(f"Testing Streamlit App: {app_entry}")

at = AppTest.from_file(str(app_entry), default_timeout=30)
at.run()

print(f"[Chunk 12] AppTest Completed successfully!")
print(f"   Exceptions Encountered: {len(at.exception)}")
if at.exception:
    for exc in at.exception:
        print(f"   Exception: {exc}")
assert len(at.exception) == 0, "Streamlit app raised an unexpected exception during headless render!"
print("\n============================================================")
print("ALL 12 PIPELINE CHUNKS VERIFIED AND WORKING FLAWLESSLY!")
print("============================================================")
